# Decoder Diverse Training — Flux-Apex-V1.flx

**Incremental Training with Crash-Safe Checkpoints**

This notebook trains the WaveDecoder on diverse, high-quality datasets to improve generation coherency.

## Key Features
- **Loads** Flux-Apex-V1.flx from HuggingFace
- **Trains** on 10,000 curated examples (5 categories × 2000)
- **Saves** back to Flux-Apex-V1.flx every 500 steps (crash-safe)
- **Uploads** to HuggingFace at each checkpoint
- **Word-boundary aware** training (improves coherency)

## Datasets (2000 each)
| Category | Purpose |
|----------|--------|
| Assistancy | Helpful, structured responses |
| Dialogues | Natural conversation flow |
| Language Understanding | Q&A, comprehension |
| Storytelling | Narrative coherence |
| Reasoning | Step-by-step logic |

## Hardware
- **Target:** Kaggle T4 (16GB VRAM)
- **Runtime:** ~80 minutes
- **Checkpoint:** Every 500 steps → upload to HF (~5 checkpoints)

---

## Cell 1: Clone/Pull FLUX Repository

In [ ]:
%%time
import os
from pathlib import Path

REPO_URL = 'https://github.com/Unseengap/FLUX.git'
ROOT = Path('/kaggle/working/FLUX') if Path('/kaggle').exists() else Path('/content/FLUX')

if ROOT.exists():
    print('  Pulling latest...')
    !cd {ROOT} && git pull
else:
    print('  Cloning repo...')
    !git clone {REPO_URL} {ROOT}

os.chdir(ROOT)
print(f'  Working dir: {os.getcwd()}')

# Install dependencies
!pip install -q -e . 2>/dev/null || pip install -q -r requirements.txt
!pip install -q huggingface_hub datasets transformers

print('  ✓ Dependencies installed')

## Cell 2: Setup Paths & Logger

In [ ]:
import sys
from pathlib import Path

ROOT = Path('/kaggle/working/FLUX') if Path('/kaggle').exists() else Path('/content/FLUX')
PHASES_DIR = ROOT / 'phases'
CHECKPOINT_DIR = ROOT / 'checkpoints'
CHECKPOINT_DIR.mkdir(exist_ok=True)

# Add all phase paths
for phase_name in ['phase1', 'phase2', 'phase3', 'phase4', 'phase5', 'phase6', 'phase7', 'phase8', 'phase8_8', 'phase8_9', 'phase9']:
    p = PHASES_DIR / phase_name
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from flux_utils import PhaseLogger, get_device, HF_REPO_ID, _resolve_hf_token

log = PhaseLogger(phase=13)  # New phase for decoder training
log.separator("Decoder Diverse Training")

print('  ✓ Paths configured')
print(f'  ROOT: {ROOT}')
print(f'  CHECKPOINT_DIR: {CHECKPOINT_DIR}')

## Cell 3: Hardware & Secrets

In [ ]:
import torch
import gc

log.cell_start("Cell 3 — Hardware & Secrets")

device = get_device()
print(f'  Device: {device}')

if device == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  GPU: {gpu_name} ({vram:.1f} GB)')

# Load HF token
hf_token = _resolve_hf_token()
if hf_token:
    print('  ✓ HF_TOKEN loaded')
else:
    print('  ⚠ HF_TOKEN not found — checkpoints will only save locally')

log.cell_end("Cell 3 — Hardware & Secrets", "PASS")

## Cell 4: Download & Load Flux-Apex-V1.flx

In [ ]:
from huggingface_hub import hf_hub_download
from datetime import datetime

log.cell_start("Cell 4 — Load Flux-Apex-V1.flx")

FLX_NAME = 'Flux-Apex-V1.flx'
FLX_HF_PATH = f'checkpoints/{FLX_NAME}'
flx_path = CHECKPOINT_DIR / FLX_NAME

# Always download fresh to get latest version
print(f'  Downloading {FLX_NAME} from HuggingFace...')
try:
    downloaded = hf_hub_download(
        repo_id=HF_REPO_ID,
        filename=FLX_HF_PATH,
        local_dir=str(ROOT),
        token=hf_token,
        force_download=False,  # Use cache if exists
    )
    size_mb = Path(downloaded).stat().st_size / 1e6
    print(f'  ✓ Downloaded {FLX_NAME} ({size_mb:.1f} MB)')
except Exception as e:
    if flx_path.exists():
        print(f'  ⚠ Download failed, using local: {e}')
        size_mb = flx_path.stat().st_size / 1e6
    else:
        raise RuntimeError(f"Cannot get {FLX_NAME}: {e}")

# Load the full archive
print(f'\n  Loading {FLX_NAME}...')
apex = torch.load(str(flx_path), map_location='cpu', weights_only=False)

print(f'  ✓ Loaded: {FLX_NAME}')
print(f'    Version: {apex.get("version", "unknown")}')
print(f'    Phase: {apex.get("phase", "unknown")}')
print(f'    Top-level keys: {len(apex)}')

# Check decoder exists
if 'decoder' not in apex:
    raise KeyError("No decoder found in Flux-Apex-V1.flx!")

decoder_info = apex['decoder']
decoder_config = decoder_info.get('config', {})
print(f'\n  Decoder config:')
print(f'    Hidden dim: {decoder_config.get("hidden_dim", 1024)}')
print(f'    Layers: {decoder_config.get("num_layers", 4)}')

log.metric("Model version", apex.get('version'))
log.cell_end("Cell 4 — Load Flux-Apex-V1.flx", "PASS")

## Cell 5: Build WaveDecoder from Apex State

In [ ]:
log.cell_start("Cell 5 — Build WaveDecoder")

# Import WaveDecoder
from wave_decoder import WaveDecoder
from cse import ContinuousSemanticEncoder

# Build CSE (frozen, for encoding)
# CSE has sensible defaults - just load state_dict
cse = ContinuousSemanticEncoder()
cse.load_state_dict(apex['cse']['state_dict'])
cse = cse.to(device).eval()
for p in cse.parameters():
    p.requires_grad = False
print(f'  ✓ CSE loaded and frozen')

# Build WaveDecoder
decoder_config = apex['decoder'].get('config', {
    'wave_dim': 432,
    'field_features': 512,
    'embed_dim': 256,
    'hidden_dim': 1024,
    'num_layers': 4,
    'num_heads': 16,
    'vocab_size': 256,
})

decoder = WaveDecoder(
    wave_dim=decoder_config.get('wave_dim', 432),
    field_features=decoder_config.get('field_features', 512),
    embed_dim=decoder_config.get('embed_dim', 256),
    hidden_dim=decoder_config.get('hidden_dim', 1024),
    num_layers=decoder_config.get('num_layers', 4),
    num_heads=decoder_config.get('num_heads', 16),
    vocab_size=decoder_config.get('vocab_size', 256),
)

# Load weights
decoder_state = apex['decoder']['state_dict']
# Clean _orig_mod. prefix if present
cleaned_state = {k.replace('_orig_mod.', ''): v for k, v in decoder_state.items()}
decoder.load_state_dict(cleaned_state)
decoder = decoder.to(device)

n_params = sum(p.numel() for p in decoder.parameters())
print(f'  ✓ WaveDecoder loaded: {n_params:,} parameters')

# Get field features for context (use mean of field state)
field_state = apex['field']['state_dict']['state']  # [96, 96, 96, 512]
field_mean = field_state.mean(dim=(0, 1, 2))  # [512]
print(f'  ✓ Field mean computed for context: {field_mean.shape}')

log.metric("Decoder params", f"{n_params:,}")
log.cell_end("Cell 5 — Build WaveDecoder", "PASS")

## Cell 6: Load Diverse Datasets (500 each)

In [ ]:
log.cell_start("Cell 6 — Load Diverse Datasets")

from datasets import load_dataset
import random

random.seed(42)

print("\n  Loading 5 diverse datasets (2000 each)...")
print("  " + "="*50)

all_texts = []
dataset_sources = {}

def clear_mem():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ─────────────────────────────────────────────
# 1. ASSISTANCY (500) — Helpful responses
# ─────────────────────────────────────────────
print("\n  [1/5] Assistancy (OpenAssistant)...")
try:
    oasst = load_dataset("OpenAssistant/oasst1", split="train")
    assist_texts = []
    for item in oasst:
        if item.get('text') and len(item['text']) > 50 and len(item['text']) < 500:
            assist_texts.append(item['text'])
    assist_texts = random.sample(assist_texts, min(2000, len(assist_texts)))
    all_texts.extend(assist_texts)
    dataset_sources['assistancy'] = len(assist_texts)
    del oasst
    clear_mem()
    print(f"       ✓ {len(assist_texts)} samples")
except Exception as e:
    print(f"       ⚠ Failed: {e}")
    dataset_sources['assistancy'] = 0

# ─────────────────────────────────────────────
# 2. DIALOGUES (2000) — Natural conversations
# ─────────────────────────────────────────────
print("\n  [2/5] Dialogues (Anthropic HH-RLHF)...")
try:
    hh = load_dataset("Anthropic/hh-rlhf", split="train")
    dialog_texts = []
    for item in hh:
        text = item.get('chosen', '')
        if text and 100 < len(text) < 800:
            dialog_texts.append(text)
        if len(dialog_texts) >= 5000:  # Stop early to save memory
            break
    dialog_texts = random.sample(dialog_texts, min(2000, len(dialog_texts)))
    all_texts.extend(dialog_texts)
    dataset_sources['dialogues'] = len(dialog_texts)
    del hh
    clear_mem()
    print(f"       ✓ {len(dialog_texts)} samples")
except Exception as e:
    print(f"       ⚠ Failed: {e}")
    dataset_sources['dialogues'] = 0

# ─────────────────────────────────────────────
# 3. LANGUAGE UNDERSTANDING (500) — Q&A
# ─────────────────────────────────────────────
print("\n  [3/5] Language Understanding (SQuAD)...")
try:
    squad = load_dataset("squad", split="train")
    qa_texts = []
    for item in squad:
        q = item.get('question', '')
        a = item.get('answers', {}).get('text', [''])[0] if item.get('answers') else ''
        if q and a:
            text = f"Question: {q}\nAnswer: {a}"
            if 30 < len(text) < 400:
                qa_texts.append(text)
    qa_texts = random.sample(qa_texts, min(2000, len(qa_texts)))
    all_texts.extend(qa_texts)
    dataset_sources['qa'] = len(qa_texts)
    del squad
    clear_mem()
    print(f"       ✓ {len(qa_texts)} samples")
except Exception as e:
    print(f"       ⚠ Failed: {e}")
    dataset_sources['qa'] = 0

# ─────────────────────────────────────────────
# 4. STORYTELLING (500) — Narrative coherence
# ─────────────────────────────────────────────
print("\n  [4/5] Storytelling (WritingPrompts)...")
try:
    wp = load_dataset("euclaise/writingprompts", split="train")
    story_texts = []
    for item in wp:
        story = item.get('story', '') or item.get('text', '')
        if story and 100 < len(story) < 800:
            story_texts.append(story[:600])  # Truncate long stories
    story_texts = random.sample(story_texts, min(2000, len(story_texts)))
    all_texts.extend(story_texts)
    dataset_sources['storytelling'] = len(story_texts)
    del wp
    clear_mem()
    print(f"       ✓ {len(story_texts)} samples")
except Exception as e:
    print(f"       ⚠ Failed: {e}")
    # Fallback to Dolly
    try:
        dolly = load_dataset("databricks/databricks-dolly-15k", split="train")
        story_texts = []
        for item in dolly:
            if item.get('category') == 'creative_writing':
                text = f"{item.get('instruction', '')}\n{item.get('response', '')}"
                if 100 < len(text) < 800:
                    story_texts.append(text)
        story_texts = random.sample(story_texts, min(2000, len(story_texts)))
        all_texts.extend(story_texts)
        dataset_sources['storytelling'] = len(story_texts)
        del dolly
        clear_mem()
        print(f"       ✓ {len(story_texts)} samples (from Dolly fallback)")
    except Exception as e2:
        print(f"       ⚠ Fallback also failed: {e2}")
        dataset_sources['storytelling'] = 0

# ─────────────────────────────────────────────
# 5. REASONING (500) — Step-by-step logic
# ─────────────────────────────────────────────
print("\n  [5/5] Reasoning (GSM8K)...")
try:
    gsm = load_dataset("gsm8k", "main", split="train")
    reason_texts = []
    for item in gsm:
        q = item.get('question', '')
        a = item.get('answer', '')
        if q and a:
            text = f"Problem: {q}\n\nSolution: {a}"
            if len(text) < 800:
                reason_texts.append(text)
    reason_texts = random.sample(reason_texts, min(2000, len(reason_texts)))
    all_texts.extend(reason_texts)
    dataset_sources['reasoning'] = len(reason_texts)
    del gsm
    clear_mem()
    print(f"       ✓ {len(reason_texts)} samples")
except Exception as e:
    print(f"       ⚠ Failed: {e}")
    dataset_sources['reasoning'] = 0

# Shuffle all texts
random.shuffle(all_texts)

print("\n  " + "="*50)
print("  DATASET SUMMARY:")
total = 0
for name, count in dataset_sources.items():
    print(f"    {name}: {count}")
    total += count
print(f"  {'─'*30}")
print(f"    TOTAL: {total} samples")
print("  " + "="*50)

log.metric("Total samples", total)
log.cell_end("Cell 6 — Load Diverse Datasets", "PASS")

## Cell 7: Training Configuration

In [ ]:
log.cell_start("Cell 7 — Training Configuration")

import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# Training config
BATCH_SIZE = 1  # Process one at a time (variable length)
GRAD_ACCUM = 4  # Effective batch = 4
MAX_SEQ_LEN = 384  # Bytes per sample
LR = 2e-4
WARMUP_STEPS = 100
CHECKPOINT_EVERY = 500  # Save to Apex every 500 steps

# Optimizer
optimizer = AdamW(decoder.parameters(), lr=LR, weight_decay=0.01)

# Scheduler (cosine decay)
total_steps = len(all_texts)
scheduler = CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=LR/10)

print(f"  Training Configuration:")
print(f"    Total samples: {len(all_texts)}")
print(f"    Gradient accumulation: {GRAD_ACCUM}")
print(f"    Effective batch size: {GRAD_ACCUM}")
print(f"    Max seq length: {MAX_SEQ_LEN} bytes")
print(f"    Learning rate: {LR}")
print(f"    Checkpoint every: {CHECKPOINT_EVERY} steps")
print(f"    Estimated checkpoints: {total_steps // CHECKPOINT_EVERY}")

log.cell_end("Cell 7 — Training Configuration", "PASS")

## Cell 8: Checkpoint Save/Upload Function

In [ ]:
log.cell_start("Cell 8 — Checkpoint Functions")

from huggingface_hub import HfApi

def save_decoder_to_apex(decoder, apex_dict, step, metrics, path, upload=True):
    """
    Save updated decoder back into Apex archive and optionally upload.
    
    This ensures crash-safe training — each checkpoint updates the model.
    """
    # Update decoder state in apex dict
    apex_dict['decoder']['state_dict'] = decoder.state_dict()
    apex_dict['decoder']['config']['training_steps'] = apex_dict['decoder'].get('config', {}).get('training_steps', 4500) + step
    
    # Update metadata
    apex_dict['metadata'] = apex_dict.get('metadata', {})
    apex_dict['metadata']['last_decoder_training'] = datetime.now().isoformat()
    apex_dict['metadata']['decoder_training_step'] = step
    apex_dict['metadata']['decoder_training_metrics'] = metrics
    apex_dict['modified'] = True
    apex_dict['modified_components'] = apex_dict.get('modified_components', []) 
    if 'decoder' not in apex_dict['modified_components']:
        apex_dict['modified_components'].append('decoder')
    
    # Save locally
    torch.save(apex_dict, str(path))
    size_mb = path.stat().st_size / 1e6
    print(f"    ✓ Saved locally: {path.name} ({size_mb:.1f} MB)")
    
    # Upload to HuggingFace
    if upload and hf_token:
        try:
            api = HfApi(token=hf_token)
            api.upload_file(
                path_or_fileobj=str(path),
                path_in_repo=f'checkpoints/{path.name}',
                repo_id=HF_REPO_ID,
                commit_message=f'Decoder training step {step} — loss={metrics.get("loss", 0):.4f}',
            )
            print(f"    ✓ Uploaded to HuggingFace Hub")
        except Exception as e:
            print(f"    ⚠ Upload failed: {e}")
    
    return True


def test_generation(decoder, cse, field_mean, prompt, max_len=60):
    """Quick generation test."""
    decoder.eval()
    with torch.no_grad():
        wave = cse.encode(prompt)
        wave_seq = wave.full.to(device)
        output = decoder.generate(
            wave_sequence=wave_seq,
            field_features=field_mean.to(device),
            max_length=max_len,
            temperature=0.7,
        )
    decoder.train()
    return output.decode('utf-8', errors='replace')

print("  ✓ Checkpoint functions defined")
log.cell_end("Cell 8 — Checkpoint Functions", "PASS")

## Cell 9: Training Loop (Word-Boundary Aware)

In [ ]:
log.cell_start("Cell 9 — Training Loop")

import time

print("\n" + "="*60)
print("  DECODER DIVERSE TRAINING")
print("  Crash-safe: saves to Flux-Apex-V1.flx every 500 steps")
print("="*60 + "\n")

# Training state
global_step = 0
accum_loss = 0.0
total_loss = 0.0
best_loss = float('inf')
start_time = time.time()

# Word boundary loss weighting
# Increase loss weight at word boundaries (spaces) to improve coherency
WORD_BOUNDARY_WEIGHT = 2.0  # 2x weight on space predictions

decoder.train()
optimizer.zero_grad()

for idx, text in enumerate(all_texts):
    try:
        # Encode text to bytes
        text_bytes = text.encode('utf-8', errors='replace')[:MAX_SEQ_LEN]
        if len(text_bytes) < 10:
            continue
        
        target = torch.tensor(list(text_bytes), dtype=torch.long, device=device)
        
        # Encode with CSE (frozen)
        with torch.no_grad():
            wave = cse.encode(text[:MAX_SEQ_LEN])
            wave_seq = wave.full.to(device)
        
        # Forward pass
        logits = decoder(
            target_bytes=target,
            wave_sequence=wave_seq,
            field_features=field_mean.to(device),
        )
        
        # Compute loss with word-boundary weighting
        # Find space positions (byte 32)
        weights = torch.ones(len(target), device=device)
        space_mask = (target == 32)  # ASCII space
        weights[space_mask] = WORD_BOUNDARY_WEIGHT
        
        # Weighted cross-entropy
        loss_per_token = F.cross_entropy(logits, target, reduction='none')
        loss = (loss_per_token * weights).mean()
        
        # Backward with accumulation
        loss = loss / GRAD_ACCUM
        loss.backward()
        accum_loss += loss.item() * GRAD_ACCUM
        
        # Step optimizer
        if (idx + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(decoder.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            
            global_step += 1
            total_loss += accum_loss
            avg_loss = total_loss / global_step
            
            # Progress update
            if global_step % 50 == 0:
                elapsed = time.time() - start_time
                rate = global_step / elapsed
                eta = (len(all_texts) // GRAD_ACCUM - global_step) / rate / 60
                print(f"  Step {global_step}: loss={accum_loss:.4f} avg={avg_loss:.4f} lr={scheduler.get_last_lr()[0]:.2e} ETA={eta:.1f}min")
            
            # Checkpoint
            if global_step % CHECKPOINT_EVERY == 0:
                print(f"\n  ════ CHECKPOINT at step {global_step} ════")
                
                # Test generation
                test_prompt = "The meaning of life is"
                gen = test_generation(decoder, cse, field_mean, test_prompt)
                print(f"    Test: '{test_prompt}' → '{gen}'")
                
                # Save to Apex
                metrics = {
                    'step': global_step,
                    'loss': accum_loss,
                    'avg_loss': avg_loss,
                    'samples_seen': idx + 1,
                }
                save_decoder_to_apex(decoder, apex, global_step, metrics, flx_path, upload=True)
                
                if accum_loss < best_loss:
                    best_loss = accum_loss
                    print(f"    ★ New best loss: {best_loss:.4f}")
                
                print()
            
            accum_loss = 0.0
    
    except Exception as e:
        print(f"  ⚠ Error at sample {idx}: {e}")
        continue

# Final checkpoint
print(f"\n  ════ FINAL CHECKPOINT ════")
total_time = time.time() - start_time
final_avg_loss = total_loss / max(1, global_step)

metrics = {
    'total_steps': global_step,
    'final_loss': accum_loss if accum_loss > 0 else final_avg_loss,
    'avg_loss': final_avg_loss,
    'best_loss': best_loss,
    'total_time_seconds': total_time,
    'samples_trained': len(all_texts),
}
save_decoder_to_apex(decoder, apex, global_step, metrics, flx_path, upload=True)

print(f"\n  Training complete!")
print(f"    Total steps: {global_step}")
print(f"    Final avg loss: {final_avg_loss:.4f}")
print(f"    Best loss: {best_loss:.4f}")
print(f"    Time: {total_time/60:.1f} minutes")

log.metric("Total steps", global_step)
log.metric("Final loss", f"{final_avg_loss:.4f}")
log.cell_end("Cell 9 — Training Loop", "PASS")

## Cell 10: Generation Quality Test

In [ ]:
log.cell_start("Cell 10 — Generation Quality Test")

print("\n" + "="*60)
print("  POST-TRAINING GENERATION TEST")
print("="*60 + "\n")

test_prompts = [
    "The meaning of life is",
    "Hello! How can I help you today?",
    "Once upon a time in a magical forest,",
    "Question: What is the capital of France?\nAnswer:",
    "To solve this problem, we need to",
]

decoder.eval()
for prompt in test_prompts:
    gen = test_generation(decoder, cse, field_mean, prompt, max_len=80)
    print(f"  Prompt: '{prompt}'")
    print(f"  Output: '{gen}'")
    print()

log.cell_end("Cell 10 — Generation Quality Test", "PASS")

## Cell 11: Summary

In [ ]:
log.separator("Training Complete")

print("""
╔════════════════════════════════════════════════════════════════════╗
║           DECODER DIVERSE TRAINING COMPLETE                        ║
╠════════════════════════════════════════════════════════════════════╣
║                                                                    ║
║  DATASETS TRAINED:                                                 ║
║  ├── Assistancy (OpenAssistant)                                   ║
║  ├── Dialogues (DailyDialog)                                      ║
║  ├── Language Understanding (SQuAD)                               ║
║  ├── Storytelling (WritingPrompts/Dolly)                          ║
║  └── Reasoning (GSM8K)                                            ║
║                                                                    ║
║  KEY FEATURES:                                                     ║
║  ✓ Word-boundary aware training (2x weight on spaces)             ║
║  ✓ Crash-safe checkpoints every 500 steps                         ║
║  ✓ Auto-upload to HuggingFace Hub                                 ║
║  ✓ Updated Flux-Apex-V1.flx with improved decoder                 ║
║                                                                    ║
║  OUTPUT: Flux-Apex-V1.flx (updated)                               ║
╚════════════════════════════════════════════════════════════════════╝
""")

print(f"\nFinal Statistics:")
print(f"  Samples trained: {len(all_texts)}")
print(f"  Total steps: {global_step}")
print(f"  Final avg loss: {final_avg_loss:.4f}")
print(f"  Checkpoints saved: {global_step // CHECKPOINT_EVERY + 1}")
print(f"  Training time: {total_time/60:.1f} minutes")

log.success("Decoder diverse training complete!")